# Phase 6 — Planning, Memory & Context
**Goal:** Introduce conversation memory, multi-step reasoning, and demonstrate improved multi-turn quality over the stateless Phase 2 baseline.

---

In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv("../.env")

from agent.agent import build_agent, run_agent_turn
from agent.memory import AgentMemory

executor = build_agent(verbose=False)
memory = AgentMemory(window_k=10)
print("Agent with memory ready.")

Pinecone index 'taskflow-kb' already exists.
Index 'taskflow-kb' has 52 vectors — skipping upsert.
Agent with memory ready.


## 6.1 Memory Architecture

| Memory Type | Implementation | Retention | Storage |
|---|---|---|---|
| **Short-term** | `ConversationBufferWindowMemory(k=10)` | Last 10 turns in session | In-memory (RAM) |
| **Session issue tracking** | `unresolved_turns` counter | Resets on resolution or manual reset | In-memory |
| **Interaction log** | JSONL file (PII-masked) | Persists across sessions | `logs/agent_interactions.jsonl` |

**Reset behaviour:** Memory is cleared when `memory.reset()` is called (e.g., user ends session or logs out).

## 6.2 Multi-Turn Conversation with Memory
The agent correctly references earlier context instead of treating each message as independent.

In [2]:
conversation = [
    "I'm on the Free plan and I can't create any more projects.",
    "How much would it cost to upgrade and get unlimited projects?",
    "Does that plan also include the Gantt chart?",
    "And what about the Slack integration? Is that included too?",
    "OK great. How do I actually upgrade my plan?",
]

print("Multi-turn conversation with memory:")
print("=" * 60)
memory.reset()

for i, msg in enumerate(conversation, 1):
    resp = run_agent_turn(executor, memory, msg)
    print(f"Turn {i}")
    print(f"  USER : {msg}")
    print(f"  AGENT: {resp[:300]}")
    print()

Multi-turn conversation with memory:


Google Sheets init failed: Expecting value: line 1 column 1 (char 0)


Turn 1
  USER : I'm on the Free plan and I can't create any more projects.
  AGENT: It looks like you're encountering the project limit for the Free plan on TaskFlow Pro. The Free plan allows for a maximum of 3 projects. To create more projects, you'll need to upgrade to the Pro plan or higher. You can do this by visiting the billing section at [app.taskflowpro.com/billing](https:/

Turn 2
  USER : How much would it cost to upgrade and get unlimited projects?
  AGENT: To upgrade to the Pro plan, which offers unlimited projects, the cost is $12 per user per month if billed monthly. If you choose to be billed annually, the cost is reduced to $10 per user per month, offering a 17% savings.

If you need further assistance with upgrading or have any other questions, f

Turn 3
  USER : Does that plan also include the Gantt chart?
  AGENT: Yes, the Pro plan includes access to the Gantt chart feature. This allows you to view tasks in a timeline format, showing durations and dependencies. If yo

## 6.3 Contrast: Without Memory (Phase 2 Baseline)
The stateless rule-based agent cannot follow the same conversation.

In [3]:
# Re-import the Phase 2 rule-based agent for comparison
import re, uuid
from datetime import datetime, timezone

RULES_P2 = [
    {"keywords": ["project", "free plan", "can't create"], "response": "Free plan supports max 3 projects. Upgrade to Pro for unlimited."},
    {"keywords": ["cost", "price", "upgrade", "pro"], "response": "Pro is $12/user/month. Business is $25/user/month."},
    {"keywords": ["gantt"], "response": "Gantt chart is available on Pro and Business plans."},
    {"keywords": ["slack", "integration"], "response": "Slack integration is available on Pro and Business plans."},
    {"keywords": ["how", "upgrade"], "response": "Upgrade at app.taskflowpro.com/billing."},
]
FALLBACK_P2 = "I'm sorry, I don't have a ready answer for that."

def rule_p2(user_input):
    lower = user_input.lower()
    for rule in RULES_P2:
        if any(kw in lower for kw in rule["keywords"]):
            return rule["response"]
    return FALLBACK_P2

print("Same conversation — stateless rule-based agent (Phase 2):")
print("=" * 60)
for i, msg in enumerate(conversation, 1):
    resp = rule_p2(msg)
    print(f"Turn {i}")
    print(f"  USER : {msg}")
    print(f"  AGENT: {resp}")
    print()

print(">>> Notice: Turn 3 ('Does that plan...') fails because 'that plan' has no context in the stateless agent.")

Same conversation — stateless rule-based agent (Phase 2):
Turn 1
  USER : I'm on the Free plan and I can't create any more projects.
  AGENT: Free plan supports max 3 projects. Upgrade to Pro for unlimited.

Turn 2
  USER : How much would it cost to upgrade and get unlimited projects?
  AGENT: Free plan supports max 3 projects. Upgrade to Pro for unlimited.

Turn 3
  USER : Does that plan also include the Gantt chart?
  AGENT: Gantt chart is available on Pro and Business plans.

Turn 4
  USER : And what about the Slack integration? Is that included too?
  AGENT: Slack integration is available on Pro and Business plans.

Turn 5
  USER : OK great. How do I actually upgrade my plan?
  AGENT: Pro is $12/user/month. Business is $25/user/month.

>>> Notice: Turn 3 ('Does that plan...') fails because 'that plan' has no context in the stateless agent.


## 6.4 Multi-Step Reasoning / Planning
A complex issue requiring the agent to plan multiple steps before responding.

In [4]:
memory.reset()
complex_query = (
    "I'm a team admin. My team is on the Free plan with 3 projects. "
    "Two of my colleagues can't see the Gantt chart, and yesterday I got an ERR-403 "
    "when I tried to set up the GitHub integration. What should I do?"
)

print("Complex multi-issue query (requires planning):")
print("=" * 60)
print(f"USER: {complex_query}")
resp = run_agent_turn(executor, memory, complex_query)
print(f"AGENT: {resp}")

Complex multi-issue query (requires planning):
USER: I'm a team admin. My team is on the Free plan with 3 projects. Two of my colleagues can't see the Gantt chart, and yesterday I got an ERR-403 when I tried to set up the GitHub integration. What should I do?
AGENT: It looks like there are two separate issues here:

1. **Gantt Chart Visibility**: The Gantt chart feature is only available on the Pro and Business plans. Since your team is on the Free plan, this feature won't be accessible to your colleagues. To use the Gantt chart, you would need to upgrade to at least the Pro plan. [Source: TaskFlow Pro Knowledge Base]

2. **ERR-403 When Setting Up GitHub Integration**: The ERR-403 error indicates that you do not have permission to perform this action. This could be because the GitHub integration is only available on the Pro and Business plans. Additionally, you might need to ensure that you have the necessary permissions within your GitHub organization to authorize the integration. [So

## 6.5 Memory Retention and Reset Rules

| Rule | Behaviour |
|---|---|
| **Within-session retention** | Last 10 turns kept in `ConversationBufferWindowMemory`. Older turns drop off automatically. |
| **Session end** | `memory.reset()` clears all in-memory state. Fresh start for next session. |
| **Persistent log** | Every turn is written to `logs/agent_interactions.jsonl` (PII-masked). This log persists across sessions and is used for evaluation in Phase 9. |
| **Unresolved counter** | Resets to 0 whenever the agent successfully resolves an issue. Auto-escalation fires at 2 consecutive unresolved turns. |

In [5]:
print("Memory state BEFORE reset:")
history = memory.get_history()
print(f"  {len(history)} messages in short-term memory")

memory.reset()

print("Memory state AFTER reset:")
history = memory.get_history()
print(f"  {len(history)} messages in short-term memory")

Memory state BEFORE reset:
  2 messages in short-term memory
Memory state AFTER reset:
  0 messages in short-term memory


## 6.6 Summary

| Capability | Phase 2 (Baseline) | Phase 6 (With Memory) |
|---|---|---|
| Multi-turn follow-up ("Does that plan also...") | ❌ No context retained | ✅ Resolves pronouns and references |
| Progressive problem narrowing | ❌ Each turn independent | ✅ Builds on prior turns |
| Multi-issue complex query | ❌ Matches first keyword only | ✅ Addresses all sub-issues |
| Session history logging | ❌ No logs | ✅ PII-masked JSONL log |